# Sentiment Analysis: NLTK VADER vs. Hugging Face Transformers

In this notebook, we:
1. Load an unclean dataset containing customer feedback (`unclean_customer_feedback_200.csv`).
2. Perform data cleaning (removing HTML tags, URLs, emails, emojis, and noise).
3. Apply **NLTK VADER** to analyze sentiment.
4. Apply **Hugging Face Transformer (DistilBERT)** for deep-learning sentiment analysis.
5. Compare evaluation metrics and save processed results.

In [15]:
import pandas as pd
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Download VADER lexicon
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

## 1. Load Unclean Customer Feedback Dataset

In [16]:
# Load dataset
df = pd.read_csv('unclean_customer_feedback_200.csv')

print(f"Dataset Shape: {df.shape}")
df.head(10)

Dataset Shape: (200, 5)


,Review_ID,Customer_Name,Email,Feedback,Expected_Sentiment
0,1001,Barbara Garcia,barbara.garcia70@yahoo.com,<div>Loved the sleek design</div> and amazing ...,Positive
1,1002,Lisa Lee,lisa.lee52@hotmail.com,The item arrived yesterday. I haven't used it ...,Neutral
2,1003,Andrew Hill,andrew.hill67@outlook.com,Worst product EVER!!! Completely useless & a t...,Negative
3,1004,Jennifer Thomas,jennifer.thomas32@yahoo.com,<div class='review'><b>I have used the product...,Neutral
4,1005,Michael Wilson,michael.wilson44@gmail.com,I can't recommend this product. Very disappoin...,Negative
5,1006,Michelle Roberts,michelle.roberts71@yahoo.com,Worst product EVER!!! Completely useless & a t...,Negative
6,1007,Michelle Roberts,michelle.roberts29@gmail.com,Absolutely fantastic product!!! Highly recomme...,Positive
7,1008,Michael Wilson,michael.wilson91@gmail.com,"<div class='review'><b>Fast delivery, great pa...",Positive
8,1009,Daniel Martinez,daniel.martinez9@gmail.com,So happy I bought this!!! Best decision EVER 😊...,Positive
9,1010,Nancy Hall,nancy.hall10@yahoo.com,So happy I bought this!!! Best decision EVER 😊😊😊,Positive


## 2. Text Preprocessing & Cleaning

We clean the raw `Feedback` text by removing:
- **HTML tags** (`<div>`, `<b>`, `<h1>`, etc.)
- **URLs & Links** (`http://...`, `https://...`)
- **Email Addresses** (`support@company.com`)
- **Emojis & Special Non-ASCII Characters** (`😊`, `❤️`, etc.)
- **Extra Whitespaces**

In [17]:
def clean_feedback_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 2. Remove URLs / Links
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # 3. Remove Email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # 4. Remove Emojis & Non-ASCII characters
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    
    # 5. Remove noise words like 'Contact us at for details'
    text = re.sub(r'Contact us at\s*for details\.?', '', text, flags=re.IGNORECASE)
    
    # 6. Normalize extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning
df['Cleaned_Feedback'] = df['Feedback'].apply(clean_feedback_text)

# Compare before and after cleaning
df[['Feedback', 'Cleaned_Feedback']].head(10)

,Feedback,Cleaned_Feedback
0,<div>Loved the sleek design</div> and amazing ...,Loved the sleek design and amazing functionali...
1,The item arrived yesterday. I haven't used it ...,The item arrived yesterday. I haven't used it ...
2,Worst product EVER!!! Completely useless & a t...,Worst product EVER!!! Completely useless & a t...
3,<div class='review'><b>I have used the product...,I have used the product once so far. It seems ...
4,I can't recommend this product. Very disappoin...,I can't recommend this product. Very disappoin...
5,Worst product EVER!!! Completely useless & a t...,Worst product EVER!!! Completely useless & a t...
6,Absolutely fantastic product!!! Highly recomme...,Absolutely fantastic product!!! Highly recomme...
7,"<div class='review'><b>Fast delivery, great pa...","Fast delivery, great packaging, and excellent ..."
8,So happy I bought this!!! Best decision EVER 😊...,So happy I bought this!!! Best decision EVER M...
9,So happy I bought this!!! Best decision EVER 😊😊😊,So happy I bought this!!! Best decision EVER


## 3. Sentiment Analysis using NLTK VADER

In [18]:
sid = SentimentIntensityAnalyzer()

def analyze_vader_sentiment(text):
    scores = sid.polarity_scores(text)
    compound = scores['compound']
    
    if compound >= 0.05:
        return 'Positive'
    elif compound <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

# Compute compound scores & predicted sentiment
df['Compound_Score'] = df['Cleaned_Feedback'].apply(lambda x: sid.polarity_scores(x)['compound'])
df['VADER_Sentiment'] = df['Cleaned_Feedback'].apply(analyze_vader_sentiment)

df[['Cleaned_Feedback', 'Compound_Score', 'VADER_Sentiment', 'Expected_Sentiment']].head(10)

,Cleaned_Feedback,Compound_Score,VADER_Sentiment,Expected_Sentiment
0,Loved the sleek design and amazing functionali...,0.8617,Positive,Positive
1,The item arrived yesterday. I haven't used it ...,0.0000,Neutral,Neutral
2,Worst product EVER!!! Completely useless & a t...,-0.9086,Negative,Negative
3,I have used the product once so far. It seems ...,0.2263,Positive,Neutral
4,I can't recommend this product. Very disappoin...,-0.6811,Negative,Negative
5,Worst product EVER!!! Completely useless & a t...,-0.9086,Negative,Negative
6,Absolutely fantastic product!!! Highly recomme...,0.7822,Positive,Positive
7,"Fast delivery, great packaging, and excellent ...",0.8650,Positive,Positive
8,So happy I bought this!!! Best decision EVER M...,0.8770,Positive,Positive
9,So happy I bought this!!! Best decision EVER,0.8770,Positive,Positive


## 4. Evaluation & Accuracy Metrics

In [19]:
accuracy = accuracy_score(df['Expected_Sentiment'], df['VADER_Sentiment'])
print(f"Accuracy Score: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(df['Expected_Sentiment'], df['VADER_Sentiment']))

Accuracy Score: 89.00%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.93      0.95      0.94        80
     Neutral       1.00      0.55      0.71        40
    Positive       0.83      1.00      0.91        80

    accuracy                           0.89       200
   macro avg       0.92      0.83      0.85       200
weighted avg       0.90      0.89      0.88       200



## 4. Sentiment Analysis using Hugging Face Transformers (DistilBERT)

We load a pre-trained Transformer model (`distilbert-base-uncased-finetuned-sst-2-english`) to analyze sentiment using deep contextual embeddings.

In [20]:
# Initialize 3-Class Hugging Face Transformer pipeline
transformer_analyzer = pipeline('sentiment-analysis', model='cardiffnlp/twitter-roberta-base-sentiment-latest')

# Get predictions
transformer_results = transformer_analyzer(df['Cleaned_Feedback'].tolist())

# Store predictions (map labels: positive -> Positive, neutral -> Neutral, negative -> Negative)
df['Transformer_Sentiment'] = [r['label'].capitalize() for r in transformer_results]
df['Transformer_Confidence'] = [round(r['score'], 4) for r in transformer_results]

df[['Cleaned_Feedback', 'VADER_Sentiment', 'Transformer_Sentiment', 'Transformer_Confidence', 'Expected_Sentiment']].head(10)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Cleaned_Feedback,VADER_Sentiment,Transformer_Sentiment,Transformer_Confidence,Expected_Sentiment
0,Loved the sleek design and amazing functionali...,Positive,Positive,0.9869,Positive
1,The item arrived yesterday. I haven't used it ...,Neutral,Neutral,0.6378,Neutral
2,Worst product EVER!!! Completely useless & a t...,Negative,Negative,0.9595,Negative
3,I have used the product once so far. It seems ...,Positive,Positive,0.7227,Neutral
4,I can't recommend this product. Very disappoin...,Negative,Negative,0.9393,Negative
5,Worst product EVER!!! Completely useless & a t...,Negative,Negative,0.9595,Negative
6,Absolutely fantastic product!!! Highly recomme...,Positive,Positive,0.9823,Positive
7,"Fast delivery, great packaging, and excellent ...",Positive,Positive,0.9840,Positive
8,So happy I bought this!!! Best decision EVER M...,Positive,Positive,0.9897,Positive
9,So happy I bought this!!! Best decision EVER,Positive,Positive,0.9877,Positive


## 5. Model Evaluation and Comparison

In [21]:
vader_acc = accuracy_score(df['Expected_Sentiment'], df['VADER_Sentiment'])
trans_acc = accuracy_score(df['Expected_Sentiment'], df['Transformer_Sentiment'])

print(f"VADER Accuracy: {vader_acc * 100:.2f}%")
print(f"Transformer Accuracy: {trans_acc * 100:.2f}%")

print("\n--- VADER Classification Report ---")
print(classification_report(df['Expected_Sentiment'], df['VADER_Sentiment']))

print("\n--- Transformer Classification Report ---")
print(classification_report(df['Expected_Sentiment'], df['Transformer_Sentiment']))

VADER Accuracy: 89.00%
Transformer Accuracy: 87.50%

--- VADER Classification Report ---
              precision    recall  f1-score   support

    Negative       0.93      0.95      0.94        80
     Neutral       1.00      0.55      0.71        40
    Positive       0.83      1.00      0.91        80

    accuracy                           0.89       200
   macro avg       0.92      0.83      0.85       200
weighted avg       0.90      0.89      0.88       200


--- Transformer Classification Report ---
              precision    recall  f1-score   support

    Negative       0.93      1.00      0.96        80
     Neutral       1.00      0.38      0.55        40
    Positive       0.81      1.00      0.89        80

    accuracy                           0.88       200
   macro avg       0.91      0.79      0.80       200
weighted avg       0.90      0.88      0.85       200



## 6. Save Cleaned and Analyzed Results

In [8]:
output_file = 'cleaned_customer_feedback_analyzed.csv'
df.to_csv(output_file, index=False)
print(f"Successfully saved processed results with VADER and Transformer outputs to '{output_file}'!")

Successfully saved processed results with VADER and Transformer outputs to 'cleaned_customer_feedback_analyzed.csv'!
